# Imports.

In [ ]:
import os
import requests
import pandas as pd
import geopandas as gpd

# Télécharger les positions des écoles en Ile-de-France.

In [ ]:
url_ecoles = "https://data.iledefrance.fr/api/explore/v2.1/catalog/datasets/les_etablissements_d_enseignement_des_1er_et_2d_degres_en_idf/exports/csv?lang=fr&timezone=Europe%2FBerlin&use_labels=true&delimiter=%3B"

# Chemin du répertoire pour mettre le fichier csv d'emplacement des écoles.
folder_path_ecoles = os.path.join(
    "..", "data", "raw"
)

# Vérifier et créer le répertoire de destination s'il n'existe pas.

if not os.path.exists(folder_path_ecoles):
    os.makedirs(folder_path_ecoles)
    print(f"Répertoire créé : {folder_path_ecoles}")

In [ ]:
# Définir une fonction pour télécharger les sources.

def telecharger_csv(url, destination, fichier):
    # Chemin complet du fichier.
    filepath = os.path.join(destination, fichier)
    try:
        print(f"Téléchargement en cours depuis : {url}")

        # Effectuer une requête HTTP.
        reponse = requests.get(url, timeout=10)

        # Erreur si le statut HTTP n'est pas bon (404, 500).
        reponse.raise_for_status()

        # Ecrire le contenu dans un fichier (en mode binaire).
        with open(filepath, "wb") as f:
            f.write(reponse.content)

        print(f"Fichier enregistré sous : {filepath}")

    except requests.exceptions.RequestException as e:
        print(f"Erreur lors du téléchargement : {e}")


In [ ]:
if __name__ == "__main__":
    # Télécharger les fichiers de validations.

    url = url_ecoles
    destination = folder_path_ecoles
    fichier = "ecoles.csv"

    telecharger_csv(url, destination, fichier)

# Explorer et nettoyer les données des écoles.

In [ ]:
filepath_ecoles = os.path.join(folder_path_ecoles, "ecoles.csv")
ecoles_df = pd.read_csv(filepath_ecoles, sep=";")

In [ ]:
ecoles_df.head()

In [ ]:
ecoles_df.shape

In [ ]:
ecoles_df.columns

In [ ]:
# Supprimer les colonnes non utiles.
colonnes_supprimer = [
    'Adresse : désignation de la voie', 
    'Adresse : 5e ligne',
    'Adresse : boite postale ou course spéciale', 
    "Localité d'acheminement",
    'Libellé de la commune',
    'Géolocalisation : coordonnée X', 
    'Géolocalisation : coordonnée Y',
    'EPSG', 
    'Appariement par IGN',
    'localisation par IGN', 
    "Code nature de l'UAI",
    "Libellé de la nature de l'UAI",
    "Etat de l'établissement",
    "Libellé de l'état de l'établissement",
    'Code INSEE du département ou de la collectivité',
    'Code INSEE de la région', 
    "Code de l'académie",
    'Code INSEE de la commune',
    'Libellé du département ou de la collectivité', 
    'Libellé de la région',
    "Libellé de l'académie", 
    'Code du type de contrat', 'Libellé du type de contrat',
    'Code de la tutelle ministérielle', 
    'Libellé de la tutelle',
    'Date de rentrée pédagogique', 
    "Sigle de l'UAI", 
    'Identifiants RNB',
    'Latitude et longitude WGS84'
    
]

ecoles_df = ecoles_df.drop(columns=colonnes_supprimer)

In [ ]:
ecoles_df.head()

In [ ]:
ecoles_df.info()

In [ ]:
ecoles_df.describe()

In [ ]:
ecoles_df.isna().sum()

In [ ]:
# Suppimer les valeurs manquantes qui correspondent à "Latitude et longitude WGS84".
ecoles_df = ecoles_df.dropna(subset=["Latitude WGS84"])

In [ ]:
ecoles_df.isna().sum()

In [ ]:
ecoles_df.shape

# Données liste des stations.

In [ ]:
filepath = os.path.join("..", "data", "processed", "stations.csv")
stations_df = pd.read_csv(filepath)

In [ ]:
stations_df.head()

# Cross-join avec GeoPandas.

In [ ]:
# Convertir les DataFrames en GeoDataFrames.
gdf_stations = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df['longitude'], stations_df['latitude']),
    crs="EPSG:4326"
)

gdf_ecoles = gpd.GeoDataFrame(
    ecoles_df,
    geometry=gpd.points_from_xy(ecoles_df['Longitude WGS84'], ecoles_df['Latitude WGS84']),
    crs="EPSG:4326"
)


In [ ]:
# Projeter en Lambert-93 (EPSG:2154) pour travailler en mètre.
gdf_stations = gdf_stations.to_crs(epsg=2154)
gdf_ecoles = gdf_ecoles.to_crs(epsg=2154)


In [ ]:
# Créer un rayon de recherche autour des stations.
distance_max = 500 # en mètres.
gdf_stations['zone_recherche'] = gdf_stations.geometry.buffer(distance_max)


In [ ]:
# Définir le buffer comme la géométrie active pour la jointure.
gdf_stations_buffer = gdf_stations.set_geometry('zone_recherche')


In [ ]:
# Jointure spatiale pour trouver les écoles qui sont dans le rayon de la station.
df_resultat = gpd.sjoin(
    gdf_ecoles,
    gdf_stations_buffer[['id_ref_zdc','nom_zdc', 'res_com', 'zone_recherche']],
    how='inner',
    predicate='within'
)

In [ ]:
df_resultat.head(10)

In [ ]:
# Exemple de filtrage par nom école.
df_resultat[df_resultat['Appellation officielle'].str.contains('Stanislas', na=False)]

In [ ]:
# Exemple de filtrage par nom station.
df_resultat[df_resultat['nom_zdc'].str.contains('Falguière', na=False)]